# West-Med Skytruth oil-slick detections

Query the [Cerulean](https://cerulean.skytruth.org) slick catalogue for the
western Mediterranean (lon -6 to 20, lat 35 to 45) over June 2023 and save the
detections as an annotated Parquet file.

Skytruth is an **open** source — no credentials needed. Detections can be
sparse: widen the region or time window if a query returns none. Files are
written under `notebooks/cache/` (git-ignored).

In [ ]:
from pathlib import Path

from IPython.display import display

import collekt
from collekt.core.config import get_config


def _repo_root(marker: str = "pyproject.toml") -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    return here


OUTPUT_ROOT = _repo_root() / "notebooks" / "cache" / "westmed_skytruth"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

config = get_config(
    overrides={
        "output": {"root": str(OUTPUT_ROOT)},
        "sources": {
            "skytruth": {
                "kind": "skytruth",
                "variable_groups": ["oil_slick"],
                "path": "skytruth",
                "filename_pattern": "skytruth_{start:%Y%m%d}_{end:%Y%m%d}_{bbox_hash}.parquet",
                "limit": 1000,
            }
        },
    }
)

request = collekt.Request(
    region=collekt.Region.from_bbox((-6.0, 20.0, 35.0, 45.0)),
    start="2023-06-01",
    end="2023-06-30",
)
request.as_dict()

## Download

In [ ]:
fetcher = collekt.Fetcher(request, config=config, progress=lambda source, message: print(f"[{source}] {message}"))
result = fetcher.download()

print(result.summary)
for item in result.results:
    print(f"  {item.source}: {item.status.value} -> {item.path or item.message}")

## Inspect the detections

The adapter writes an annotated Parquet file; read it back with polars.

In [ ]:
import polars as pl

slicks = pl.read_parquet(result.files[0]) if result.files else None
if slicks is None:
    print("No detections for this query; widen the region or time window.")
else:
    print(f"{slicks.height} detections, {slicks.width} columns")
    columns = [c for c in ("id", "slick_timestamp", "machine_confidence", "area", "length") if c in slicks.columns]
    display(slicks.select(columns).head(10))

## Machine-confidence distribution

In [ ]:
import matplotlib.pyplot as plt

if slicks is not None and "machine_confidence" in slicks.columns:
    values = slicks["machine_confidence"].drop_nulls().to_list()
    plt.figure(figsize=(8, 4))
    plt.hist(values, bins=20)
    plt.xlabel("machine_confidence")
    plt.ylabel("detections")
    plt.title("Skytruth slick machine confidence")
    plt.show()